Dataset Cleaning

In [1]:
from pathlib import Path 
import pandas as pd 

notebook_dir = Path.cwd()

project_root = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir

data_path = project_root / "heart_disease_uci.csv"
df = pd.read_csv(data_path)

In [2]:
df_clean = df.copy()

df_clean = df_clean[df_clean["dataset"] == "Cleveland"]
df_clean = df_clean.dropna(subset=["trestbps", "chol", "fbs", "thalch", "exang", "oldpeak", "slope", "ca", "thal"])


# Binary Variables: sex, fbs, exang
df_clean["sex"] = df_clean["sex"].map({"Female": 0, "Male": 1})
df_clean["fbs"] = df_clean["fbs"].astype(int) #True/False = 1/0
df_clean["exang"] = df_clean["exang"].astype(int)

# Nominal Variables: dataset, cp, restecg, thal
df_clean = pd.get_dummies(df_clean, columns=["cp", "restecg", "thal"], drop_first=True)
df_clean = df_clean.drop(columns="dataset")

# Ordinal Variables: slope, ca
slope_order = {"upsloping": 0, "flat": 1, "downsloping": 2}
df_clean["slope" ] = df_clean["slope"].map(slope_order)

df_clean = df_clean.drop(columns=["id"])

df_clean["target"] = (df_clean["num"] > 0).astype(int)
df_clean = df_clean.drop(columns=["num"])

def audit(df_clean):
    summary = pd.DataFrame({
        "dtype": df_clean.dtypes,
        "nulls": df_clean.isnull().sum(),
        "null_%": (df_clean.isnull().mean() * 100).round(2),
        "unique": df_clean.nunique(),
        "sample": df_clean.iloc[0]
    })
    return summary

print(audit(df_clean))
df_clean.shape


                            dtype  nulls  null_%  unique sample
age                         int64      0     0.0      41     63
sex                         int64      0     0.0       2      1
trestbps                  float64      0     0.0      50  145.0
chol                      float64      0     0.0     152  233.0
fbs                         int64      0     0.0       2      1
thalch                    float64      0     0.0      91  150.0
exang                       int64      0     0.0       2      0
oldpeak                   float64      0     0.0      40    2.3
slope                       int64      0     0.0       3      2
ca                        float64      0     0.0       4    0.0
cp_atypical angina           bool      0     0.0       2  False
cp_non-anginal               bool      0     0.0       2  False
cp_typical angina            bool      0     0.0       2   True
restecg_normal               bool      0     0.0       2  False
restecg_st-t abnormality     bool      0

(297, 18)

Preprocessing

In [3]:
# Train/test split
from pathlib import Path
from sklearn.model_selection import train_test_split

X = df_clean.drop(columns=["target"])
y = df_clean["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

project_root = Path.cwd().parent
processed_dir = project_root / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(processed_dir / "X_train.csv", index=False)
X_test.to_csv(processed_dir / "X_test.csv", index=False)
y_train.to_csv(processed_dir / "y_train.csv", index=False)
y_test.to_csv(processed_dir / "y_test.csv", index=False)

print(f"Saved: X_train {X_train.shape}, X_test {X_test.shape}")

Saved: X_train (237, 17), X_test (60, 17)
